# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [ ]:
import pandas as pd

file = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(file)

# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Strip whitespace from object columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()


In [ ]:
# low total_claim_amount (e.g., below $1,000) 

low_claim_yes_df = df[
    (df['total_claim_amount'] < 1000) &
    (df['response'].str.lower() == 'yes')
]

low_claim_yes_df.head()
low_claim_yes_df.shape


In [ ]:
# have a response "Yes" to the last marketing campaign

yes_df = df[df['response'].str.lower() == 'yes']


In [ ]:
# Using the original Dataframe, analyze:
# the average monthly_premium and/or customer lifetime value by policy_type and gender for customers who responded "Yes", and
# compare these insights to total_claim_amount patterns, and discuss which segments appear most profitable or low-risk for the compa

premium_clv_summary = yes_df.groupby(['policy_type', 'gender'])[
    ['monthly_premium_auto', 'customer_lifetime_value', 'total_claim_amount']
].mean().round(2)

premium_clv_summary


In [ ]:
# Some policy types (e.g., Corporate Auto or Personal Auto) show higher CLV, meaning they are more profitable long‑term.
# Gender differences may appear — sometimes males show slightly higher CLV, sometimes females.
# Monthly premium patterns show which segments pay more per month.
# Compare CLV vs total_claim_amount:
	* High CLV + low claim amount = best customers
	* Low CLV + high claim amount = high‑risk customers

In [ ]:
# Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

customers_per_state = df['state'].value_counts()
customers_per_state


In [ ]:
large_states = customers_per_state[customers_per_state > 500]
large_states


In [ ]:
clv_stats = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(
    ['max', 'min', 'median']
).round(2)

clv_stats


In [ ]:
# Higher education levels (Bachelor, Master, Doctorate) often show higher CLV, meaning more profitable customers.
# Gender differences may appear — sometimes males have slightly higher max CLV, sometimes females.
# Median CLV is the most stable indicator of typical customer value.
# Education + gender segmentation helps identify which groups respond best to marketing and maintain long‑term relationships.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here

In [8]:
import pandas as pd

file = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(file)

df.columns = df.columns.str.lower().str.replace(' ', '_')


In [9]:
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])
df['month'] = df['effective_to_date'].dt.month_name()


/var/folders/t1/g1ptmxln4nx5k_rz9978lf340000gn/T/ipykernel_20490/633240684.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])


In [11]:
policies_state_month = df.pivot_table(
    values='customer',        # any column works, we just count rows
    index='state',
    columns='month',
    aggfunc='count',
    fill_value=0
)

policies_state_month

month,February,January
state,,
Arizona,929,1008
California,1634,1918
Nevada,442,551
Oregon,1344,1565
Washington,425,463


In [12]:
policies_state_month


month,February,January
state,,
Arizona,929,1008
California,1634,1918
Nevada,442,551
Oregon,1344,1565
Washington,425,463


In [13]:
state_counts = df['state'].value_counts()
state_counts


state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64

In [14]:
top3_states = state_counts.head(3).index
top3_states


Index(['California', 'Oregon', 'Arizona'], dtype='str', name='state')

In [ ]:
# California and Oregon are the strongest markets and Arizona is mid-tier market. 

In [15]:
df_top3 = df[df['state'].isin(top3_states)]


In [16]:
top3_policies = df_top3.groupby(['state', 'month'])['customer'].count().reset_index()
top3_policies.rename(columns={'customer': 'policies_sold'}, inplace=True)


In [17]:
top3_policies


,state,month,policies_sold
0,Arizona,February,929
1,Arizona,January,1008
2,California,February,1634
3,California,January,1918
4,Oregon,February,1344
5,Oregon,January,1565


In [18]:
df['response_yes'] = (df['response'].str.lower() == 'yes').astype(int)


In [19]:
response_rate = df.groupby('sales_channel')['response_yes'].mean().round(3)
response_rate


sales_channel
Agent          0.180
Branch         0.108
Call Center    0.103
Web            0.109
Name: response_yes, dtype: float64

In [20]:
response_rate_df = response_rate.reset_index()
response_rate_df.rename(columns={'response_yes': 'response_rate'}, inplace=True)


In [21]:
response_long = response_rate_df.melt(
    id_vars='sales_channel',
    var_name='metric',
    value_name='value'
)


In [22]:
response_long


,sales_channel,metric,value
0,Agent,response_rate,0.180
1,Branch,response_rate,0.108
2,Call Center,response_rate,0.103
3,Web,response_rate,0.109


In [ ]:
# Agent channel has the highest response rate 
# Branch , web and call center channels perform similarly